# Model Pipeline - Fashion Forward Forecasting

This notebook demonstrates how to build, train, and evaluate an end-to-end ML pipeline
for predicting customer product recommendations based on reviews.

The pipeline handles:
- Numeric features (scaling)
- Categorical features (one-hot encoding)
- Text features (TF-IDF vectorization)
- Hyperparameter tuning via RandomizedSearchCV
- Model persistence using joblib

## Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    roc_auc_score
)

from src.data_processing import (
    load_data,
    prepare_features_target,
    split_data,
    get_column_types
)
from src.model import (
    build_pipeline,
    train_pipeline,
    save_pipeline,
    load_pipeline
)

# Set random seed for reproducibility
RANDOM_STATE = 27

## 1. Load and Explore Data

Load the reviews dataset from `data/raw/reviews.csv`.

In [ ]:
# Load data
df = load_data('../data/raw/reviews.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn types:")
print(df.dtypes)
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Check target distribution
print("Target distribution:")
print(df['Recommended IND'].value_counts(normalize=True))

## 2. Prepare Features and Target

In [ ]:
# Separate features and target
X, y = prepare_features_target(df, target_column='Recommended IND')

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nLabels: {y.unique()}")

In [ ]:
# Get column types
col_types = get_column_types(df)

print("Column types identified:")
for col_type, columns in col_types.items():
    print(f"  {col_type}: {columns}")

## 3. Split Data

In [ ]:
# Split into train and test sets
X_train, X_test, y_train, y_test = split_data(
    X, y,
    test_size=0.1,
    random_state=RANDOM_STATE
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 4. Build the Pipeline

Create a pipeline that:
1. Preprocesses numeric features (imputation + scaling)
2. Preprocesses categorical features (imputation + one-hot encoding)
3. Preprocesses text features (combine + clean + TF-IDF)
4. Trains a LogisticRegression classifier

In [ ]:
# Build the pipeline
pipeline = build_pipeline(
    numeric_features=col_types['numeric'],
    categorical_features=col_types['categorical'],
    text_features=col_types['text'],
    max_tfidf_features=5000,
    random_state=RANDOM_STATE
)

print("Pipeline structure:")
print(pipeline)

## 5. Train with Hyperparameter Search

Use RandomizedSearchCV to find the best hyperparameters.

In [ ]:
# Train the pipeline with hyperparameter search
search = train_pipeline(
    pipeline=pipeline,
    X_train=X_train,
    y_train=y_train,
    n_iter=10,
    cv=3,
    scoring='f1',
    random_state=RANDOM_STATE,
    verbose=2
)

print(f"\nBest cross-validation F1 score: {search.best_score_:.4f}")

## 6. Inspect Best Parameters

In [ ]:
# Display best parameters
print("Best parameters found:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")

In [ ]:
# View cross-validation results
cv_results = pd.DataFrame(search.cv_results_)
cv_results = cv_results.sort_values('rank_test_score')

# Display top 5 configurations
display_cols = ['rank_test_score', 'mean_test_score', 'std_test_score', 'mean_train_score']
param_cols = [col for col in cv_results.columns if col.startswith('param_')]
cv_results[display_cols + param_cols].head(5)

## 7. Evaluate on Holdout Test Set

In [ ]:
# Get the best estimator
best_model = search.best_estimator_

# Make predictions on test set
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Calculate metrics
print("=" * 50)
print("TEST SET EVALUATION")
print("=" * 50)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

In [ ]:
# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Recommended', 'Recommended']))

In [ ]:
# Confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

## 8. Save the Trained Pipeline

In [ ]:
# Save the best model
save_pipeline(search, filepath='../models/model_v1.joblib')

## 9. Verify Model Loading

In [ ]:
# Load the saved model and verify it works
loaded_model = load_pipeline('../models/model_v1.joblib')

# Test prediction
y_pred_loaded = loaded_model.predict(X_test)

# Verify predictions match
assert np.array_equal(y_pred, y_pred_loaded), "Predictions don't match!"
print("Model loaded successfully and predictions verified!")

## Summary

This notebook demonstrated:

1. **Data Loading**: Used `src.data_processing` module to load and prepare the dataset
2. **Feature Engineering**: Built a `ColumnTransformer` that handles numeric, categorical, and text data
3. **Pipeline Construction**: Created an sklearn `Pipeline` combining preprocessing and classification
4. **Hyperparameter Tuning**: Used `RandomizedSearchCV` for efficient hyperparameter search
5. **Evaluation**: Assessed model performance on a holdout test set
6. **Persistence**: Saved and loaded the trained model using joblib

The pipeline is reproducible (random seeds set) and follows best practices for ML workflows.